
#### 05 — Train ML Classifier

##### Purpose

This notebook trains the first supervised machine-learning classifier for the Support Ticket NLP project.

It:

- reads the persisted modeling dataset
- reuses the existing train/validation/test assignments
- fits TF-IDF using training text only
- trains a Logistic Regression multiclass classifier
- generates validation predictions
- performs basic training validation
- keeps the test set untouched for final evaluation in Notebook 06

The notebook is independently runnable.

It does not depend on variables created in Notebook 04.

##### 1. 

``` text 

nlp_modeling_dataset
        ↓
dataset_split
        ↓
 ┌────────────┬──────────────┬────────────┐
 ↓            ↓              ↓
Train      Validation        Test
 ↓            ↓              ↓
TF-IDF fit    TF-IDF          untouched
+ transform   transform
 ↓            ↓
X_train    X_validation
 ↓
Logistic Regression
 ↓
model.fit()
 ↓
trained classifier
 ↓
validation predictions

```

The test set remains reserved for Notebook 06.


##### 2. Why Rebuild TF-IDF Here?

Notebook 04 created:

- tfidf_vectorizer
- X_train_tfidf
- X_validation_tfidf
- X_test_tfidf

for learning and inspection.

But Notebook 05 must not depend on Notebook 04 having run first.

Therefore it independently performs:

``` text

load persisted data
        ↓
create train/validation/test views
        ↓
fit TF-IDF on train
        ↓
train classifier

```

This is one of our production design principles:

Persistent tables are dependencies. Notebook memory is not.


##### 3. Technologies

- Python
- PySpark
- pandas
- scikit-learn
- TF-IDF
- Logistic Regression
- Unity Catalog
- Delta Lake

##### 4. Imports

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from src.project_config import (
    MODELING_TABLE,
    TICKET_ID_COL,
    CLEAN_TEXT_COL,
    TARGET_COL,
    SPLIT_COL,
    EXPECTED_CATEGORIES,
    RANDOM_SEED,
    TFIDF_MAX_FEATURES,
    TFIDF_NGRAM_RANGE,
    TFIDF_MIN_DF,
    TFIDF_MAX_DF,
    BASELINE_MODEL_NAME,
)

##### 5. Verify Configuration

In [0]:
print(f"Modeling table : {MODELING_TABLE}")
print(f"Baseline model : {BASELINE_MODEL_NAME}")
print(f"Random seed    : {RANDOM_SEED}")

print(
    "Expected categories:",
    EXPECTED_CATEGORIES,
)

##### 6. Load the Modeling Dataset

In [0]:
modeling_df = spark.table(
    MODELING_TABLE
)

In [0]:
display(
    modeling_df.limit(20)
)

In [0]:
modeling_row_count = (
    modeling_df.count()
)

print(
    f"Modeling row count: "
    f"{modeling_row_count:,}"
)

In [0]:
if modeling_row_count == 0:
    raise ValueError(
        f"Modeling table contains no records: "
        f"{MODELING_TABLE}"
    )

##### 7. Validate Input Contract

In [0]:
required_columns = {
    TICKET_ID_COL,
    CLEAN_TEXT_COL,
    TARGET_COL,
    SPLIT_COL,
}

In [0]:
available_columns = set(
    modeling_df.columns
)

missing_columns = (
    required_columns
    - available_columns
)

In [0]:
if missing_columns:
    raise ValueError(
        "Missing required modeling columns: "
        f"{sorted(missing_columns)}"
    )

print(
    "Input schema validation passed."
)

##### 8. Validate Dataset Splits

In [0]:
actual_splits = {
    row[SPLIT_COL]
    for row in (
        modeling_df
        .select(SPLIT_COL)
        .distinct()
        .collect()
    )
}

expected_splits = {
    "train",
    "validation",
    "test",
}


In [0]:
if actual_splits != expected_splits:
    raise ValueError(
        "Unexpected dataset splits.\n"
        f"Expected: {sorted(expected_splits)}\n"
        f"Actual:   {sorted(actual_splits)}"
    )

print(
    "Dataset split validation passed."
)

##### 9. Validate Target Categories

In [0]:
actual_categories = {
    row[TARGET_COL]
    for row in (
        modeling_df
        .select(TARGET_COL)
        .distinct()
        .collect()
    )
}

expected_categories = set(
    EXPECTED_CATEGORIES
)

In [0]:
if actual_categories != expected_categories:
    raise ValueError(
        "Dataset categories do not match "
        "project configuration.\n"
        f"Expected: {sorted(expected_categories)}\n"
        f"Actual:   {sorted(actual_categories)}"
    )

print(
    "Category validation passed."
)

##### 10. Create Train / Validation / Test DataFrames

In [0]:
train_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "train"
    )
)

validation_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "validation"
    )
)

test_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "test"
    )
)

In [0]:
train_count = train_df.count()
validation_count = validation_df.count()
test_count = test_df.count()

print(f"Train rows      : {train_count:,}")
print(f"Validation rows : {validation_count:,}")
print(f"Test rows       : {test_count:,}")

##### 11. Verify Category Coverage in Each Split

In [0]:
split_category_df = (
    modeling_df
    .groupBy(
        SPLIT_COL,
        TARGET_COL,
    )
    .count()
    .orderBy(
        SPLIT_COL,
        TARGET_COL,
    )
)

In [0]:
display(
    split_category_df
)

##### 12. Convert Required Columns to pandas

In [0]:
train_pdf = (
    train_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

validation_pdf = (
    validation_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

test_pdf = (
    test_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

In [0]:
print(
    "Train shape      :",
    train_pdf.shape,
)

print(
    "Validation shape :",
    validation_pdf.shape,
)

print(
    "Test shape       :",
    test_pdf.shape,
)

##### 13. Separate Text and Target

In [0]:
X_train_text = (
    train_pdf[
        CLEAN_TEXT_COL
    ]
)

y_train = (
    train_pdf[
        TARGET_COL
    ]
)

In [0]:
X_validation_text = (
    validation_pdf[
        CLEAN_TEXT_COL
    ]
)

y_validation = (
    validation_pdf[
        TARGET_COL
    ]
)

In [0]:
X_test_text = (
    test_pdf[
        CLEAN_TEXT_COL
    ]
)

y_test = (
    test_pdf[
        TARGET_COL
    ]
)

##### 14. Create the TF-IDF Vectorizer

In [0]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
)

##### 15. Fit TF-IDF on Training Text Only

In [0]:
X_train_tfidf = (
    tfidf_vectorizer
    .fit_transform(
        X_train_text
    )
)

##### 16. Transform Validation Text

In [0]:
X_validation_tfidf = (
    tfidf_vectorizer
    .transform(
        X_validation_text
    )
)

##### 17. Transform Test Text

In [0]:
X_test_tfidf = (
    tfidf_vectorizer
    .transform(
        X_test_text
    )
)

##### 18. Verify Feature Shapes

In [0]:
print(
    "X_train_tfidf:",
    X_train_tfidf.shape,
)

print(
    "X_validation_tfidf:",
    X_validation_tfidf.shape,
)

print(
    "X_test_tfidf:",
    X_test_tfidf.shape,
)

In [0]:
if not (
    X_train_tfidf.shape[1]
    == X_validation_tfidf.shape[1]
    == X_test_tfidf.shape[1]
):
    raise ValueError(
        "TF-IDF feature dimensions "
        "do not match across datasets."
    )

print(
    "TF-IDF feature validation passed."
)

##### 19. Logistic Regression Connection


 Logistic Regression for Telco Churn.

The same model can work here.

Previously:

``` text

Customer features
        ↓
tenure
MonthlyCharges
Contract
...
        ↓
Logistic Regression
        ↓
Churn / No Churn

```

Here:

``` text

Ticket text
        ↓
TF-IDF
        ↓
85 numerical features
        ↓
Logistic Regression
        ↓
Billing / Cancellation /
Login / Technical

```

The classifier still receives:

X = numerical features
y = target labels

The only difference is where X came from.

##### 20. Binary vs Multiclass Logistic Regression

Telco Churn had two classes:

- 0 → No Churn
- 1 → Churn

This project has four:

- Billing
- Cancellation
- Login
- Technical

So this is: multiclass classification

Scikit-learn handles multiclass Logistic Regression directly.

##### 21. Create the Baseline Classifier

In [0]:
classifier = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
)

Why:max_iter=1000

It gives the optimization algorithm sufficient iterations to converge.

Why: random_state=RANDOM_SEED

It helps preserve reproducibility where randomness is involved.

##### 22. Train the Model

In [0]:
classifier.fit(
    X_train_tfidf,
    y_train,
)

##### 23. What Does Logistic Regression Learn Here?

Suppose terms such as:

- refund
- invoice
- payment
- charge

frequently occur in: Billing

while:

- router
- wifi
- internet
- connection

frequently occur in: Technical

Logistic Regression learns coefficients connecting those TF-IDF features with particular classes.

Conceptually:

``` text 

refund
    ↓ strong positive coefficient
Billing


router
    ↓ strong positive coefficient
Technical

```

These relationships are learned from training data.

We did not manually write rules such as:

if "refund" in ticket:
    category = "Billing"

##### 24. Inspect Learned Classes

In [0]:
model_classes = (
    classifier.classes_
)

print(
    "Learned classes:"
)

print(
    model_classes
)

In [0]:
if set(model_classes) != expected_categories:
    raise ValueError(
        "Classifier classes do not match "
        "the expected ticket categories."
    )

print(
    "Classifier class validation passed."
)

##### 25. Inspect Model Coefficient Shape

In [0]:
print(
    "Coefficient shape:",
    classifier.coef_.shape,
)

print(
    "Intercept shape:",
    classifier.intercept_.shape,
)

##### 26. Generate Training Predictions

In [0]:
train_predictions = (
    classifier.predict(
        X_train_tfidf
    )
)

##### 27. Generate Validation Predictions

In [0]:
validation_predictions = (
    classifier.predict(
        X_validation_tfidf
    )
)

##### 28. Do Not Predict the Test Set Yet

Technically we could run:

classifier.predict(
    X_test_tfidf
)

But we deliberately avoid using test results in Notebook 05.

Why?

Because the test set should remain our final unbiased evaluation dataset.

The workflow is:

``` text

TRAIN
 ↓
learn parameters

VALIDATION
 ↓
check model behavior
make model decisions

TEST
 ↓
final evaluation only

```

Notebook 06 will use the test set.

##### 29. Basic Training Metrics

In [0]:
train_accuracy = accuracy_score(
    y_train,
    train_predictions,
)

train_f1_macro = f1_score(
    y_train,
    train_predictions,
    average="macro",
    zero_division=0,
)

In [0]:
print(
    f"Training accuracy : "
    f"{train_accuracy:.4f}"
)

print(
    f"Training macro F1 : "
    f"{train_f1_macro:.4f}"
)

##### 30. Basic Validation Metrics

In [0]:
validation_accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

validation_precision_macro = precision_score(
    y_validation,
    validation_predictions,
    average="macro",
    zero_division=0,
)

validation_recall_macro = recall_score(
    y_validation,
    validation_predictions,
    average="macro",
    zero_division=0,
)

validation_f1_macro = f1_score(
    y_validation,
    validation_predictions,
    average="macro",
    zero_division=0,
)

In [0]:
print(
    f"Validation accuracy        : "
    f"{validation_accuracy:.4f}"
)

print(
    f"Validation macro precision : "
    f"{validation_precision_macro:.4f}"
)

print(
    f"Validation macro recall    : "
    f"{validation_recall_macro:.4f}"
)

print(
    f"Validation macro F1        : "
    f"{validation_f1_macro:.4f}"
)

##### 31. Why Macro Metrics?

We have four classes.

Macro averaging calculates the metric separately for each class and then takes the average:

``` text

Billing metric
        +
Cancellation metric
        +
Login metric
        +
Technical metric
        ↓
divide by 4

```

Each category receives equal importance.

This is useful because accuracy alone can hide poor performance on a smaller class.

Notebook 06 will explore this in much more detail.

##### 32. Inspect Validation Predictions

In [0]:
validation_results_df = (
    validation_pdf[
        [
            TICKET_ID_COL,
            CLEAN_TEXT_COL,
            TARGET_COL,
        ]
    ]
    .copy()
)

In [0]:
validation_results_df[
    "predicted_category"
] = validation_predictions

In [0]:
validation_results_df[
    "is_correct"
] = (
    validation_results_df[
        TARGET_COL
    ]
    == validation_results_df[
        "predicted_category"
    ]
)

In [0]:
display(
    validation_results_df
)

##### 33. Inspect Misclassified Validation Tickets

In [0]:
validation_errors_df = (
    validation_results_df[
        ~validation_results_df[
            "is_correct"
        ]
    ]
    .copy()
)

In [0]:
display(
    validation_errors_df
)

What wording confused the classifier?

Was an important word unknown?

Are two categories linguistically similar?

Is the dataset too small?

Was the ticket ambiguous?

Error analysis is an important part of NLP modeling.

##### 34. Get Prediction Probabilities

In [0]:
#Logistic Regression can also produce class probabilities:

validation_probabilities = (
    classifier.predict_proba(
        X_validation_tfidf
    )
)

In [0]:
print(
    validation_probabilities.shape
)

##### 35. Inspect One Probability Distribution

In [0]:
sample_probability_index = 0

sample_probabilities = (
    validation_probabilities[
        sample_probability_index
    ]
)

In [0]:
probability_df = pd.DataFrame(
    {
        "category": (
            classifier.classes_
        ),
        "probability": (
            sample_probabilities
        ),
    }
)

In [0]:
display(
    probability_df
    .sort_values(
        "probability",
        ascending=False,
    )
)

##### 36. Verify Probability Sum

In [0]:
probability_sum = (
    sample_probabilities.sum()
)

print(
    f"Probability sum: "
    f"{probability_sum:.6f}"
)

##### 37. Inspect Vocabulary Size

In [0]:
feature_names = (
    tfidf_vectorizer
    .get_feature_names_out()
)

vocabulary_size = len(
    feature_names
)

print(
    f"TF-IDF vocabulary size: "
    f"{vocabulary_size:,}"
)

##### 38. Inspect Most Influential Terms Per Class

In [0]:
# Because Logistic Regression is linear, its coefficients are interpretable.

top_n = 10

top_terms_by_class = {}

for class_index, class_name in enumerate(
    classifier.classes_
):

    coefficients = (
        classifier.coef_[
            class_index
        ]
    )

    top_indices = (
        np.argsort(
            coefficients
        )[-top_n:][::-1]
    )

    top_terms_by_class[
        class_name
    ] = pd.DataFrame(
        {
            "term": (
                feature_names[
                    top_indices
                ]
            ),
            "coefficient": (
                coefficients[
                    top_indices
                ]
            ),
        }
    )

In [0]:
for class_name, terms_df in (
    top_terms_by_class.items()
):

    print(
        f"\nTop terms for "
        f"{class_name}"
    )

    display(
        terms_df
    )

##### 39. Connection to Neural-Network Weights

In Neural Network

``` text

feature
 ↓
weight
 ↓
neuron

```

Here Logistic Regression also learns feature weights:

``` text

TF-IDF feature
 ↓
coefficient
 ↓
class score

```

For example:

``` text

refund TF-IDF
       ↓
Billing coefficient
       ↓
contributes to Billing score

```

So Logistic Regression is still learning: weights + bias just without hidden layers and nonlinear activations.

##### 40. Check for Training / Validation Gap

In [0]:
accuracy_gap = (
    train_accuracy
    - validation_accuracy
)

f1_gap = (
    train_f1_macro
    - validation_f1_macro
)

In [0]:
print(
    f"Accuracy gap: "
    f"{accuracy_gap:.4f}"
)

print(
    f"Macro F1 gap: "
    f"{f1_gap:.4f}"
)

A very large gap could indicate overfitting.

With only 100 total records, however, individual validation examples can move metrics significantly, so we should interpret the results cautiously.

##### 41. Why the Dataset Size Matters


current split is approximately:

- 70 training records
- 17 validation records
- 13 test records

This is intentionally a small learning dataset.

That means: one wrong validation prediction can noticeably change metrics.

For example:

16 / 17 correct
≈ 94.1%

15 / 17 correct
≈ 88.2%

So we should use this project to understand the architecture and modeling process rather than treat the metrics as production-grade evidence.

##### 42. Final Training Validation

In [0]:
assert X_train_tfidf.shape[0] == len(
    y_train
)

assert X_validation_tfidf.shape[0] == len(
    y_validation
)

assert X_test_tfidf.shape[0] == len(
    y_test
)

assert (
    X_train_tfidf.shape[1]
    == X_validation_tfidf.shape[1]
    == X_test_tfidf.shape[1]
)

assert (
    len(
        classifier.classes_
    )
    == len(
        EXPECTED_CATEGORIES
    )
)

assert (
    validation_probabilities.shape
    ==
    (
        validation_count,
        len(
            EXPECTED_CATEGORIES
        ),
    )
)

print(
    "Final model training validation passed."
)

##### 43. What We Have Now

We now have two fitted objects:

``` text

tfidf_vectorizer
        ↓
learned vocabulary + IDF

classifier
        ↓
learned Logistic Regression coefficients

```

Prediction requires both:

```

clean text
    ↓
tfidf_vectorizer.transform()
    ↓
TF-IDF vector
    ↓
classifier.predict()
    ↓
category

```

This dependency becomes important later when we package the model for MLflow and serving.

##### 44. Why We Should Eventually Package Them Together

We do not want deployed code to require callers to manually perform: TF-IDF

Instead, the final deployed model should ideally behave like:

``` text

"My router keeps restarting"
        ↓
model
        ↓
Technical

rather than:

caller manually builds
85-dimensional TF-IDF vector
        ↓
model

```

Later we will package preprocessing and classification into one inference artifact.

##### 45. What Notebook 05 Does Not Do

This notebook deliberately does not:

- perform final test evaluation
- build the final confusion matrix
- produce full classification reports
- log the final MLflow run
- register the model
- deploy the endpoint

Those responsibilities belong to later notebooks.

##### Production Design Decisions

This notebook follows these principles:

- It is independently runnable.
- It reads the persisted modeling dataset.
- Existing train/validation/test assignments are reused.
- TF-IDF is fitted on training data only.
- Validation and test use only transform().
- Logistic Regression is fitted only on training features and labels.
- Validation data is used for model inspection.
- Test labels remain untouched for final evaluation.
- Class probabilities are inspected separately from predicted labels.
- Model coefficients are inspected for interpretability.
- Configuration comes from src/project_config.py.
- No state from Notebook 04 is required.

##### Key Learnings

The NLP pipeline now connects directly to supervised machine learning:

``` text

Clean Text
    ↓
TF-IDF
    ↓
X features
    +
category
    ↓
y target
    ↓
Logistic Regression

```

Logistic Regression learns relationships between TF-IDF features and ticket categories.

For a four-class problem:

4 categories × 85 TF-IDF features

the classifier learns approximately:

4 × 85 = 340

feature coefficients, plus class intercepts.

The model is not using manually written keyword rules.

It learns the feature/category relationships from training examples.

##### Conclusion

Notebook 05 completes our first end-to-end NLP modeling path:

``` text

Raw Support Ticket
        ↓
Cleaning
        ↓
Tokenization
        ↓
Vocabulary
        ↓
Bag of Words
        ↓
TF-IDF
        ↓
Logistic Regression
        ↓
Predicted Ticket Category

```

We now have our first trained NLP classifier.

The model has been inspected using validation data, while the test dataset remains reserved for unbiased final evaluation.

##### Next Notebook

##### 06_evaluate_nlp_classifier

This will focus on:
``` text

Test Set
   ↓
final predictions
   ↓
Accuracy
Precision
Recall
F1
Classification Report
Confusion Matrix
Per-class performance
Error analysis

```